# Xeno Take-Home - Comm-Log Send Reconciliation (Workbench)

**Goal:** reproduce Finance's `target_base = 22` for merchant 501, October 2026,
Diwali campaigns - starting from the naive query and investigating the gap.

This notebook gives you the plumbing. The **analysis is yours**: each section has a
`YOUR TURN` prompt with a scaffolded query to run, inspect and modify.

---
### How to use
1. Upload `campaign.csv` and `communication_log.csv` to Google Drive.
2. Edit **only** the two paths in the config cell below.
3. Run the cells top to bottom.


## 0. Setup


In [ ]:
# =====================================================================
#  CONFIG - THE ONLY CELL YOU NEED TO EDIT
# =====================================================================

# How are you supplying the CSVs?
#   'drive'  -> mount Google Drive and read the paths below  (recommended)
#   'upload' -> browser file-picker, no Drive needed
SOURCE = 'drive'

# <<< CHANGE THESE TWO PATHS TO WHERE YOUR FILES ACTUALLY LIVE >>>
CAMPAIGN_CSV = '/content/drive/MyDrive/xeno/campaign.csv'
COMM_LOG_CSV = '/content/drive/MyDrive/xeno/communication_log.csv'

# Scope of the metric (from the assignment brief)
MERCHANT_ID        = 501
PERIOD_START       = '2026-10-01'   # inclusive
PERIOD_END         = '2026-11-01'   # EXCLUSIVE (half-open interval)
COMMUNICATION_TYPE = '2'            # '2' = Campaign
CAMPAIGN_NAME_LIKE = '%Diwali%'

# What Finance says the answer is. We do NOT use this to build the query -
# only to check ourselves at the very end.
FINANCE_TARGET_BASE = 22

print('Config set. SOURCE =', SOURCE)


In [ ]:
import pandas as pd, sqlite3, textwrap

if SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
elif SOURCE == 'upload':
    from google.colab import files
    print('Pick campaign.csv and communication_log.csv (select both at once)...')
    up = files.upload()
    for fn in up:
        low = fn.lower()
        if 'campaign' in low:
            CAMPAIGN_CSV = '/content/' + fn
        elif 'comm' in low:
            COMM_LOG_CSV = '/content/' + fn
    print('campaign  ->', CAMPAIGN_CSV)
    print('comm_log  ->', COMM_LOG_CSV)
else:
    raise ValueError("SOURCE must be 'drive' or 'upload'")


In [ ]:
# ---- Load CSVs and build an in-memory SQLite DB ----------------------
# na_values=[''] makes the empty parent_id cells real NULLs rather than
# the literal string '', which would break the recursive CTE.

campaign = pd.read_csv(CAMPAIGN_CSV, keep_default_na=False, na_values=[''])
comm_log = pd.read_csv(COMM_LOG_CSV, keep_default_na=False, na_values=[''])

# communication_type is TEXT ('2') in the source schema - force to str so
# the WHERE clause matches instead of silently returning zero rows.
comm_log['communication_type'] = comm_log['communication_type'].astype(str)
comm_log['customer_id']        = comm_log['customer_id'].astype(str)

# parent_id must be a nullable INTEGER, not float, or SQLite sees 9001.0
campaign['parent_id'] = pd.to_numeric(campaign['parent_id'], errors='coerce').astype('Int64')

con = sqlite3.connect(':memory:')
campaign.to_sql('campaign',          con, index=False, if_exists='replace')
comm_log.to_sql('communication_log', con, index=False, if_exists='replace')

def q(sql, params=()):
    """Run SQL against the in-memory DB, return a DataFrame."""
    return pd.read_sql_query(textwrap.dedent(sql), con, params=params)

def scalar(sql, params=()):
    """Run SQL, return the single value in the first cell."""
    return q(sql, params).iloc[0, 0]

print('campaign          : %3d rows' % len(campaign))
print('communication_log : %3d rows' % len(comm_log))
campaign


In [ ]:
# ---- Sanity check: did the data land intact? -------------------------
assert len(campaign) > 0 and len(comm_log) > 0, 'A CSV loaded empty - check your paths.'
assert campaign['parent_id'].isna().any(), 'Expected some NULL parent_id (chain roots).'
print('parent_id nulls (chain roots):', int(campaign['parent_id'].isna().sum()))
print('Types OK. Ready.')
comm_log.head()


---
## Phase 1 - Understand the data

Before writing anything clever, find out what you are actually holding.

> **YOUR TURN:** run these, then ask yourself - *which columns could possibly*
> *change the answer?* A column with one distinct value cannot.


In [ ]:
# Column profile: nulls and distinct values
q('''
  SELECT 'campaign' AS tbl, 'parent_id' AS col, COUNT(*) AS n,
         COUNT(parent_id) AS non_null, COUNT(DISTINCT parent_id) AS distinct_vals
  FROM campaign
  UNION ALL SELECT 'campaign','creation_status',COUNT(*),COUNT(creation_status),COUNT(DISTINCT creation_status) FROM campaign
  UNION ALL SELECT 'campaign','processing_status',COUNT(*),COUNT(processing_status),COUNT(DISTINCT processing_status) FROM campaign
  UNION ALL SELECT 'comm_log','customer_id',COUNT(*),COUNT(customer_id),COUNT(DISTINCT customer_id) FROM communication_log
  UNION ALL SELECT 'comm_log','communication_id',COUNT(*),COUNT(communication_id),COUNT(DISTINCT communication_id) FROM communication_log
  UNION ALL SELECT 'comm_log','delivery_status',COUNT(*),COUNT(delivery_status),COUNT(DISTINCT delivery_status) FROM communication_log
  UNION ALL SELECT 'comm_log','merchant_id',COUNT(*),COUNT(merchant_id),COUNT(DISTINCT merchant_id) FROM communication_log
  UNION ALL SELECT 'comm_log','channel',COUNT(*),COUNT(channel),COUNT(DISTINCT channel) FROM communication_log
''')


In [ ]:
# Categorical values, and what delivery_status actually means
display(q('SELECT creation_status, processing_status, COUNT(*) AS n FROM campaign GROUP BY 1,2'))
display(q('''SELECT delivery_status, COUNT(*) AS rows_, SUM(credit_used) AS credits
             FROM communication_log GROUP BY 1'''))
print('900 = delivered, 1100 = soft failure (retryable).')
print('Note both consume credits - i.e. a failure is still a real, billed send.')


In [ ]:
# Date range + boundary safety. Are there ANY sends outside the period?
q('''
  SELECT MIN(sent_time) AS min_sent, MAX(sent_time) AS max_sent,
         SUM(sent_time <  ?) AS before_period,
         SUM(sent_time >= ?) AS after_period,
         SUM(sent_time <> scheduled_time) AS sent_ne_scheduled
  FROM communication_log
''', (PERIOD_START, PERIOD_END))


In [ ]:
# Referential integrity: orphans, dangling parents, merchant mismatch
q('''
  SELECT
   (SELECT COUNT(*) FROM communication_log l LEFT JOIN campaign c ON c.id=l.communication_id
     WHERE c.id IS NULL) AS orphan_log_rows,
   (SELECT COUNT(*) FROM campaign c WHERE c.parent_id IS NOT NULL
     AND c.parent_id NOT IN (SELECT id FROM campaign)) AS dangling_parents,
   (SELECT COUNT(*) FROM communication_log l JOIN campaign c ON c.id=l.communication_id
     WHERE c.merchant_id <> l.merchant_id) AS merchant_mismatch
''')


In [ ]:
# Cardinality: campaign -> communication_log. Does joining multiply rows?
q('''SELECT (SELECT COUNT(*) FROM communication_log) AS before_join,
            (SELECT COUNT(*) FROM communication_log l
               JOIN campaign c ON c.id = l.communication_id) AS after_join''')


In [ ]:
# The campaign graph: who is a retry of whom?
q('''
  SELECT c.id, COALESCE(CAST(c.parent_id AS TEXT),'-') AS parent,
         (SELECT COUNT(*) FROM campaign k WHERE k.parent_id = c.id) AS n_children,
         CASE WHEN c.parent_id IS NULL
                   AND NOT EXISTS (SELECT 1 FROM campaign k WHERE k.parent_id = c.id)
              THEN 'STANDALONE' ELSE 'IN_CHAIN' END AS shape,
         c.creation_status, c.name
  FROM campaign c ORDER BY c.id
''')


---
## Phase 2 - The naive baseline

Write the query you would write if nobody had told you the number was wrong:
one `communication_log` row = one send.

> **YOUR TURN:** run it and write the result down before reading on.


In [ ]:
naive = scalar('''
  SELECT COUNT(*)
  FROM   communication_log l
  JOIN   campaign c ON c.id = l.communication_id
  WHERE  l.merchant_id        = ?
    AND  l.communication_type = ?
    AND  l.sent_time >= ? AND l.sent_time < ?
    AND  c.name LIKE ?
''', (MERCHANT_ID, COMMUNICATION_TYPE, PERIOD_START, PERIOD_END, CAMPAIGN_NAME_LIKE))

print('NAIVE target_base :', naive)
print('Finance says      :', FINANCE_TARGET_BASE)
print('GAP TO EXPLAIN    :', naive - FINANCE_TARGET_BASE)


---
## Phase 3 - Investigate the gap

Hypotheses to **test**, not assume. Each either moves the number (an
*adjustment*) or does not (a *validation* - still worth reporting honestly).


### H1 - Are the scope filters doing anything?

> **YOUR TURN:** if tightening a filter does not change the count, say so in
> your bridge as a validation. Do not dress a no-op up as a fix.


In [ ]:
display(q('''
  SELECT SUM(merchant_id <> ?)        AS wrong_merchant,
         SUM(communication_type <> ?) AS wrong_type,
         SUM(channel <> 'sms')        AS other_channel
  FROM communication_log''', (MERCHANT_ID, COMMUNICATION_TYPE)))

# Does the 'Diwali campaigns' filter exclude anything?
display(q('''SELECT SUM(name LIKE ?) AS diwali_campaigns, COUNT(*) AS all_campaigns
             FROM campaign''', (CAMPAIGN_NAME_LIKE,)))


### H2 - Campaign eligibility

The data dictionary says a campaign reports only once its **creation** workflow
has cleared *and* its **processing** has finished:

```sql
creation_status IN ('approved','aborted','resumed','stopped')
AND processing_status = 'processed'
```

> **YOUR TURN:** which campaigns fail this, and how many log rows do they carry?


In [ ]:
ELIGIBLE = """c.creation_status IN ('approved','aborted','resumed','stopped')
          AND c.processing_status = 'processed'"""

q('''
  SELECT c.id, c.name, c.creation_status, c.processing_status,
         CASE WHEN c.creation_status IN ('approved','aborted','resumed','stopped')
                   AND c.processing_status = 'processed'
              THEN 'ELIGIBLE' ELSE 'EXCLUDED' END AS gate,
         COUNT(l.id) AS log_rows
  FROM campaign c
  LEFT JOIN communication_log l ON l.communication_id = c.id
  GROUP BY c.id ORDER BY gate DESC, c.id
''')


In [ ]:
after_gate = scalar('''
  SELECT COUNT(*) FROM communication_log l
  JOIN campaign c ON c.id = l.communication_id
  WHERE ''' + ELIGIBLE)

print('after eligibility gate :', after_gate, '  (change:', after_gate - naive, ')')


### H3 - Repeated customers

> **YOUR TURN:** find customers appearing more than once. Then look carefully at
> **where** the repeat happens - same campaign, or a different one? Those are
> not the same thing, and that distinction is the crux of this exercise.


In [ ]:
print('Same customer, SAME campaign:')
display(q('''
  SELECT communication_id, customer_id, COUNT(*) AS n,
         GROUP_CONCAT(id) AS row_ids,
         GROUP_CONCAT(delivery_status) AS statuses,
         GROUP_CONCAT(sent_time) AS times
  FROM communication_log GROUP BY 1,2 HAVING COUNT(*) > 1'''))

print('Same customer, DIFFERENT campaigns:')
display(q('''
  SELECT customer_id, COUNT(DISTINCT communication_id) AS n_campaigns,
         GROUP_CONCAT(DISTINCT communication_id) AS campaigns
  FROM communication_log GROUP BY 1 HAVING COUNT(DISTINCT communication_id) > 1'''))


### H4 - First attempt at dedupe (expect this to be WRONG)

> **YOUR TURN:** the obvious fix is `COUNT(DISTINCT customer_id)`. Run it.
> If it does not match, **do not force it** - the mismatch is the clue. Work out
> exactly which customer you just destroyed, and why they were a real send.


In [ ]:
global_distinct = scalar('''
  SELECT COUNT(DISTINCT l.customer_id) FROM communication_log l
  JOIN campaign c ON c.id = l.communication_id
  WHERE ''' + ELIGIBLE)

print('global COUNT(DISTINCT customer) :', global_distinct)
print('Finance                         :', FINANCE_TARGET_BASE)
print()
if global_distinct < FINANCE_TARGET_BASE:
    print('Overshot by', FINANCE_TARGET_BASE - global_distinct, '- you deleted a real send.')
    print('Which repeated customer was NOT a retry? See the FIRST table in H3.')


### H5 - Retry chains: resolve each campaign to its root

A retry always creates a **new campaign row** pointing back via `parent_id`.
Chains can be deeper than two levels, so a single self-join is not enough -
use a recursive CTE.

> **YOUR TURN:** check the CTE returns exactly one row per campaign. More than
> that means a cycle or a fan-out bug.


In [ ]:
ROOT_CTE = '''
WITH RECURSIVE root_of(id, root_id) AS (
    SELECT id, id FROM campaign WHERE parent_id IS NULL
  UNION ALL
    SELECT c.id, r.root_id FROM campaign c JOIN root_of r ON c.parent_id = r.id
),
chain_size AS (
    SELECT root_id, COUNT(*) AS campaigns_in_chain FROM root_of GROUP BY root_id
)
'''

display(q(ROOT_CTE + '''
  SELECT r.id AS campaign_id, r.root_id, s.campaigns_in_chain,
         CASE WHEN s.campaigns_in_chain = 1 THEN 'STANDALONE' ELSE 'RETRY_CHAIN' END AS kind,
         c.creation_status, c.name
  FROM root_of r
  JOIN campaign   c ON c.id = r.id
  JOIN chain_size s ON s.root_id = r.root_id
  ORDER BY r.root_id, r.id'''))

chk = q(ROOT_CTE + ' SELECT COUNT(*) AS produced, COUNT(DISTINCT id) AS distinct_campaigns FROM root_of')
print(chk.to_string(index=False), ' | campaigns in table:', len(campaign))
assert int(chk.iloc[0,0]) == len(campaign), 'Recursion produced the wrong row count - cycle?'


---
## Phase 4 - Apply the rule and build the bridge

The counting rule from the data dictionary:

| Shape | Rule |
|---|---|
| **Retry chain** (>1 campaign in the family) | `COUNT(DISTINCT customer_id)` - several attempts to reach one customer is **one** communication |
| **Standalone** (no parent, no children) | `COUNT(*)` - every send row is its own event |

> **YOUR TURN:** check the per-communication contributions by hand before you
> trust the total.


In [ ]:
ELIGIBLE_SENDS = ROOT_CTE + ''',
eligible_sends AS (
    SELECT r.root_id, s.campaigns_in_chain, l.customer_id, l.id AS log_id
    FROM   communication_log l
    JOIN   campaign   c ON c.id      = l.communication_id
    JOIN   root_of    r ON r.id      = c.id
    JOIN   chain_size s ON s.root_id = r.root_id
    WHERE  l.merchant_id        = ?
      AND  l.communication_type = ?
      AND  l.sent_time >= ? AND l.sent_time < ?
      AND  c.name LIKE ?
      AND  ''' + ELIGIBLE + ')'

PARAMS = (MERCHANT_ID, COMMUNICATION_TYPE, PERIOD_START, PERIOD_END, CAMPAIGN_NAME_LIKE)

# Per-underlying-communication breakdown - check this by hand!
q(ELIGIBLE_SENDS + '''
  SELECT root_id AS underlying_communication,
         CASE WHEN campaigns_in_chain > 1 THEN 'RETRY_CHAIN' ELSE 'STANDALONE' END AS kind,
         COUNT(log_id)               AS eligible_rows,
         COUNT(DISTINCT customer_id) AS distinct_customers,
         CASE WHEN campaigns_in_chain > 1
              THEN COUNT(DISTINCT customer_id) ELSE COUNT(log_id) END AS contributes
  FROM eligible_sends GROUP BY root_id, campaigns_in_chain ORDER BY root_id
''', PARAMS)


In [ ]:
# Join-multiplication guard: these two numbers MUST be equal.
g = q(ELIGIBLE_SENDS + '''
      SELECT COUNT(*) AS rows_after_joins, COUNT(DISTINCT log_id) AS distinct_log_rows
      FROM eligible_sends''', PARAMS)
print(g.to_string(index=False))
assert int(g.iloc[0,0]) == int(g.iloc[0,1]), 'JOIN FAN-OUT - your joins are duplicating log rows.'
print('No fan-out.')


## Phase 5 - Final query


In [ ]:
FINAL_SQL = ELIGIBLE_SENDS + '''
, per_communication AS (
    SELECT root_id,
           CASE WHEN campaigns_in_chain > 1
                THEN COUNT(DISTINCT customer_id)   -- retry chain: collapse attempts
                ELSE COUNT(log_id)                 -- standalone: every send counts
           END AS qualifying_sends
    FROM eligible_sends
    GROUP BY root_id, campaigns_in_chain
)
SELECT SUM(qualifying_sends) AS target_base FROM per_communication
'''

final = scalar(FINAL_SQL, PARAMS)
print('FINAL target_base :', final)
assert final == FINANCE_TARGET_BASE, 'Got %s, Finance says %s' % (final, FINANCE_TARGET_BASE)
print('Matches Finance.')


In [ ]:
# ---- Reconciliation bridge -------------------------------------------
bridge = pd.DataFrame([
    ('0',  'Naive COUNT(*) of every send row',           naive,           '-'),
    ('1',  'Scope filters (no-op - validation)',         naive,           '0'),
    ('2',  'Drop campaigns not cleared for reporting',   after_gate,      str(after_gate - naive)),
    ('3a', 'Global COUNT(DISTINCT customer) - REJECTED', global_distinct, 'overshot'),
    ('3b', 'Dedupe within retry chains only',            final,           str(final - after_gate)),
], columns=['step', 'description', 'result', 'change'])
bridge


---
## Phase 6 - Stress-test your answer

Landing on 22 is not the same as being right. Two checks worth doing.


In [ ]:
# (a) Do OTHER interpretations also hit the target? If so you must adjudicate.
alt_table = q('''
  WITH elig AS (SELECT l.* FROM communication_log l
                JOIN campaign c ON c.id = l.communication_id
                WHERE ''' + ELIGIBLE + '''),
  root_only AS (
      SELECT c.id, COUNT(*) AS n
      FROM communication_log l JOIN campaign c ON c.id = l.communication_id
      WHERE c.parent_id IS NULL AND ''' + ELIGIBLE + '''
      GROUP BY c.id)
  SELECT 'A. naive COUNT(*)' AS interpretation,
         (SELECT COUNT(*) FROM communication_log) AS result
  UNION ALL SELECT 'B. eligible COUNT(*)',                (SELECT COUNT(*) FROM elig)
  UNION ALL SELECT 'C. global COUNT(DISTINCT customer)',  (SELECT COUNT(DISTINCT customer_id) FROM elig)
  UNION ALL SELECT 'D. distinct (campaign, customer)',    (SELECT COUNT(*) FROM (SELECT DISTINCT communication_id, customer_id FROM elig))
  UNION ALL SELECT 'E. root-only COUNT(*), no retries',   (SELECT SUM(n) FROM root_only)
  UNION ALL SELECT 'F. delivered-only (900), eligible',   (SELECT COUNT(*) FROM elig WHERE delivery_status = 900)
''')
display(alt_table)
print('G. retry-family-aware (final method):', final)
print()
print('E and F both tie the final answer here. That is a coincidence to be')
print('disproven, not evidence either one is correct - see the counterfactuals below.')


> **YOUR TURN:** if a rival interpretation ties your answer, work out *why*.
> The cell below shows the mechanism: how many eligible customers have zero,
> one, or several delivered rows.


In [ ]:
q(ROOT_CTE + ''',
per AS (
  SELECT r.root_id, l.customer_id, SUM(l.delivery_status = 900) AS delivered
  FROM communication_log l
  JOIN campaign c ON c.id = l.communication_id
  JOIN root_of  r ON r.id = c.id
  WHERE ''' + ELIGIBLE + '''
  GROUP BY r.root_id, l.customer_id)
SELECT SUM(delivered = 0) AS never_delivered,
       SUM(delivered = 1) AS exactly_one_delivery,
       SUM(delivered > 1) AS multiple_deliveries
FROM per
''')


> **YOUR TURN:** interpretation E (root-only) also ties at 22. Before trusting
> it, break it: add a customer who is only ever reached through a retry-child
> campaign, never the root. A correct chain-aware count must gain that
> customer; root-only, by construction, cannot see them.


In [ ]:
# (a2) COUNTERFACTUAL - root-only. Add one customer whose ONLY attempt is
# under a retry-CHILD campaign (never the root) - e.g. a retry audience
# expanded to include someone missed in the original blast. Runs on a
# THROWAWAY COPY - your loaded data is untouched.

cf2 = sqlite3.connect(':memory:')
con.backup(cf2)

# pick any retry-CHILD campaign (has a non-null parent_id) to attach the
# synthetic row to
child_campaign = pd.read_sql_query(
    'SELECT id FROM campaign WHERE parent_id IS NOT NULL LIMIT 1', cf2).iloc[0, 0]

cf2.execute('''INSERT INTO communication_log
               (id, merchant_id, communication_id, customer_id, communication_type,
                delivery_status, sent_time, scheduled_time, credit_used, channel)
               VALUES (99999, ?, ?, 'CF_ROOT_ONLY', ?, 900,
                       '2026-10-05 11:00:00', '2026-10-05 11:00:00', 1, 'sms')''',
           (MERCHANT_ID, int(child_campaign), COMMUNICATION_TYPE))

chain_rule_2 = pd.read_sql_query(textwrap.dedent(FINAL_SQL), cf2, params=PARAMS).iloc[0, 0]
root_only_2 = pd.read_sql_query(textwrap.dedent('''
  SELECT SUM(n) FROM (
    SELECT c.id, COUNT(*) AS n
    FROM communication_log l JOIN campaign c ON c.id = l.communication_id
    WHERE c.parent_id IS NULL AND ''' + ELIGIBLE + '''
    GROUP BY c.id)'''), cf2).iloc[0, 0]

print('Counterfactual: added CF_ROOT_ONLY under retry-child campaign %s (no root row)' % child_campaign)
print('  chain-family-aware rule :', chain_rule_2, ' (should gain the new customer)')
print('  root-only               :', root_only_2, ' (should be unchanged - misses them)')
print()
print('Diverged - root-only silently drops a customer never seen at the root.'
      if chain_rule_2 != root_only_2 else 'Still tied - investigate further.')

cf2.close()
print()
print('Original data intact:', scalar('SELECT COUNT(*) FROM communication_log'), 'rows')


In [ ]:
# (b) COUNTERFACTUAL - the decisive test.
# Make one retried customer NEVER succeed, then see which interpretations
# still agree. Runs on a THROWAWAY COPY - your loaded data is untouched.

cf = sqlite3.connect(':memory:')
con.backup(cf)

# the last successful attempt belonging to a customer who spans >1 campaign
tgt = pd.read_sql_query(textwrap.dedent('''
  SELECT l.id FROM communication_log l
  JOIN campaign c ON c.id = l.communication_id
  WHERE ''' + ELIGIBLE + ''' AND l.delivery_status = 900
    AND l.customer_id IN (SELECT customer_id FROM communication_log
                          GROUP BY customer_id
                          HAVING COUNT(DISTINCT communication_id) > 1)
  ORDER BY l.sent_time DESC LIMIT 1'''), cf).iloc[0, 0]

cf.execute('UPDATE communication_log SET delivery_status = 1100 WHERE id = ?', (int(tgt),))

chain_rule = pd.read_sql_query(textwrap.dedent(FINAL_SQL), cf, params=PARAMS).iloc[0, 0]
delivered_only = pd.read_sql_query(textwrap.dedent('''
  SELECT COUNT(*) FROM communication_log l
  JOIN campaign c ON c.id = l.communication_id
  WHERE ''' + ELIGIBLE + ''' AND l.delivery_status = 900'''), cf).iloc[0, 0]

print('Counterfactual: log row id=%s flipped 900 -> 1100 (customer never succeeds)' % tgt)
print('  chain-dedupe rule :', chain_rule)
print('  delivered-only    :', delivered_only)
print()
print('Diverged - the tie on the real data was a coincidence.'
      if chain_rule != delivered_only else 'Still tied - investigate further.')

cf.close()
print()
print('Original data intact:', scalar('SELECT COUNT(*) FROM communication_log'), 'rows')


---
## Phase 7 - Write it up

You now have everything the brief asks for:

1. **The bridge** - the `bridge` DataFrame above. Keep step 3a in it: showing a
   rejected attempt is evidence you investigated rather than guessed.
2. **The SQL** - printed below, runnable against `data/comm_log.db`.
3. **What surprised you** - candidates: a campaign carrying real sends before
   approval cleared; the same dedupe idiom being right in one campaign and wrong
   in the next; a rival interpretation tying your answer by arithmetic accident.

> **Be ready to defend every `WHERE` clause.** If you cannot name the business
> rule a condition comes from, it should not be in the query.


In [ ]:
# Final query with the parameters inlined, ready to paste into sqlite3.
sql_out = FINAL_SQL
for val in [MERCHANT_ID, COMMUNICATION_TYPE, PERIOD_START, PERIOD_END, CAMPAIGN_NAME_LIKE]:
    sql_out = sql_out.replace('?', repr(val) if isinstance(val, str) else str(val), 1)
print(sql_out)
